In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -----------------------------
# 1. CONFIG
# -----------------------------
torch.manual_seed(0)

T = 2000        # total time
D = 3           # number of target variables
EXOG = 2        # number of exogenous variables

input_len = 30
pred_len = 10


# -----------------------------
# 2. SYNTHETIC DATA
# -----------------------------
t = torch.arange(T).float()

# exogenous signals
exog = torch.stack([
    torch.sin(0.01 * t),
    torch.cos(0.02 * t)
], dim=1)  # [T, EXOG]

# multivariate target (D signals)
y = torch.stack([
    torch.sin(0.02 * t) + 0.3 * exog[:, 0],
    torch.cos(0.015 * t) + 0.3 * exog[:, 1],
    torch.sin(0.03 * t) + 0.2 * exog[:, 0] + 0.2 * exog[:, 1],
], dim=1)  # [T, D]


# -----------------------------
# 3. DATASET
# -----------------------------
class TSData(Dataset):
    def __init__(self, y, exog, input_len, pred_len):
        self.y = y
        self.exog = exog
        self.input_len = input_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.y) - self.input_len - self.pred_len

    def __getitem__(self, idx):
        i = idx

        y_past = self.y[i:i+self.input_len]                         # [T_in, D]
        x_past = self.exog[i:i+self.input_len]                     # [T_in, EXOG]

        x_future = self.exog[i+self.input_len:i+self.input_len+self.pred_len]  # [T_out, EXOG]
        y_future = self.y[i+self.input_len:i+self.input_len+self.pred_len]     # [T_out, D]

        enc = torch.cat([y_past, x_past], dim=-1)  # [T_in, D+EXOG]
        dec = x_future                              # [T_out, EXOG]

        return enc, dec, y_future


dataset = TSData(y, exog, input_len, pred_len)
loader = DataLoader(dataset, batch_size=32, shuffle=True)


# -----------------------------
# 4. MODEL
# -----------------------------
class Encoder(nn.Module):
    def __init__(self, input_size, hidden):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, batch_first=True)

    def forward(self, x):
        _, (h, c) = self.lstm(x)
        return h, c


class Decoder(nn.Module):
    def __init__(self, input_size, hidden, output_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, output_size)

    def forward(self, x, hidden):
        out, _ = self.lstm(x, hidden)
        return self.fc(out)


class Seq2Seq(nn.Module):
    def __init__(self, enc_in, dec_in, hidden, out_dim):
        super().__init__()
        self.encoder = Encoder(enc_in, hidden)
        self.decoder = Decoder(dec_in, hidden, out_dim)

    def forward(self, enc_x, dec_x):
        h, c = self.encoder(enc_x)
        return self.decoder(dec_x, (h, c))


# -----------------------------
# 5. INIT MODEL
# -----------------------------
model = Seq2Seq(
    enc_in=D + EXOG,
    dec_in=EXOG,
    hidden=64,
    out_dim=D
)

loss_fn = nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)


# -----------------------------
# 6. TRAINING LOOP
# -----------------------------
for epoch in range(10):
    total_loss = 0

    for enc, dec, y_true in loader:

        pred = model(enc, dec)

        loss = loss_fn(pred, y_true)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss:.4f}")


# -----------------------------
# 7. TEST PREDICTION
# -----------------------------
enc, dec, y_true = dataset[0]

enc = enc.unsqueeze(0)
dec = dec.unsqueeze(0)

pred = model(enc, dec)

print("\nTRUE:\n", y_true)
print("\nPRED:\n", pred.squeeze(0).detach())

Epoch 1 | Loss: 20.7910
Epoch 2 | Loss: 1.2420
Epoch 3 | Loss: 0.3406
Epoch 4 | Loss: 0.1598
Epoch 5 | Loss: 0.1144
Epoch 6 | Loss: 0.0884
Epoch 7 | Loss: 0.0754
Epoch 8 | Loss: 0.0626
Epoch 9 | Loss: 0.0623
Epoch 10 | Loss: 0.0468

TRUE:
 tensor([[0.6533, 1.1480, 1.0075],
        [0.6726, 1.1380, 1.0254],
        [0.6916, 1.1276, 1.0425],
        [0.7103, 1.1170, 1.0588],
        [0.7288, 1.1060, 1.0743],
        [0.7471, 1.0948, 1.0890],
        [0.7651, 1.0833, 1.1028],
        [0.7828, 1.0714, 1.1157],
        [0.8002, 1.0594, 1.1278],
        [0.8173, 1.0470, 1.1390]])

PRED:
 tensor([[0.6489, 1.1468, 0.9679],
        [0.6813, 1.1530, 1.0331],
        [0.6988, 1.1281, 1.0613],
        [0.7123, 1.1032, 1.0763],
        [0.7258, 1.0828, 1.0874],
        [0.7405, 1.0659, 1.0978],
        [0.7567, 1.0508, 1.1084],
        [0.7743, 1.0365, 1.1192],
        [0.7930, 1.0222, 1.1299],
        [0.8124, 1.0078, 1.1402]])


In [3]:
D

3

In [4]:
EXOG

2

In [14]:
dec.shape

torch.Size([1, 10, 2])

In [15]:
pred.shape

torch.Size([1, 10, 3])

In [3]:
import torch
import torch.nn as nn

# -------------------
# Encoder
# -------------------
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers, batch_first=True)

    def forward(self, src):
        # src: [batch_size, src_len]
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell


# -------------------
# Decoder
# -------------------
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, input, hidden, cell):
        # input: [batch_size]
        input = input.unsqueeze(1)  # [batch_size, 1]
        embedded = self.embedding(input)

        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))

        return prediction, hidden, cell


# -------------------
# Seq2Seq Model
# -------------------
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.fc_out.out_features

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        hidden, cell = self.encoder(src)

        input = trg[:, 0]  # <sos>

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t] = output

            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)

            input = trg[:, t] if teacher_force else top1

        return outputs

In [4]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: [1, batch, hidden_dim]
        # encoder_outputs: [batch, src_len, hidden_dim]

        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]

        hidden = hidden[-1].unsqueeze(1).repeat(1, src_len, 1)

        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)

        return torch.softmax(attention, dim=1)

In [5]:
INPUT_DIM = 10000
OUTPUT_DIM = 10000
EMB_DIM = 256
HIDDEN_DIM = 512
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(INPUT_DIM, EMB_DIM, HIDDEN_DIM)
decoder = Decoder(OUTPUT_DIM, EMB_DIM, HIDDEN_DIM)

model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

src = torch.randint(0, INPUT_DIM, (32, 10)).to(DEVICE)
trg = torch.randint(0, OUTPUT_DIM, (32, 12)).to(DEVICE)

output = model(src, trg)
print(output.shape)  # [32, 12, OUTPUT_DIM]

torch.Size([32, 12, 10000])


In [1]:
import torch
import torch.nn as nn

class LSTMForecaster(nn.Module):
    def __init__(self, d_in, d_act, hidden_size, d_out):
        super().__init__()

        # encoder reads past observations
        self.enc_lstm = nn.LSTM(input_size=d_in, hidden_size=hidden_size, batch_first=True)

        # decoder reads [prev_y, actuator]
        self.dec_lstm = nn.LSTM(input_size=d_out + d_act, hidden_size=hidden_size, batch_first=True)

        self.fc = nn.Linear(hidden_size, d_out)

    def encode(self, x_input):
        # x_input: [B, T_in, D]
        _, (h, c) = self.enc_lstm(x_input)
        return h, c

    def decode_step(self, x, state):
        # x: [B, 1, D_out + D_act]
        out, state = self.dec_lstm(x, state)
        y = self.fc(out[:, 0])  # [B, D_out]
        return y, state
        

In [2]:
def train_forward(model, x_input, actuator, y_true):
    """
    x_input:   [B, T_in, D]
    actuator:  [B, T_out, D_act]
    y_true:    [B, T_out, D_out]
    """

    B, T_out, _ = y_true.shape

    h, c = model.encode(x_input)

    # start token = last input state (common choice)
    y_prev = x_input[:, -1]

    preds = []

    for t in range(T_out):

        # teacher forcing: use ground truth previous output
        y_in = y_true[:, t] if t == 0 else y_true[:, t - 1]

        dec_in = torch.cat([y_in, actuator[:, t]], dim=-1).unsqueeze(1)

        y_t, (h, c) = model.decode_step(dec_in, (h, c))

        preds.append(y_t)

    return torch.stack(preds, dim=1)

In [3]:
def evaluate(model, x_input, actuator):
    """
    No access to y_true here.
    """

    B, T_out, _ = actuator.shape

    h, c = model.encode(x_input)

    # start from last observed input
    y_prev = x_input[:, -1]

    preds = []

    for t in range(T_out):

        dec_in = torch.cat([y_prev, actuator[:, t]], dim=-1).unsqueeze(1)

        y_t, (h, c) = model.decode_step(dec_in, (h, c))

        preds.append(y_t)

        # IMPORTANT: feed prediction back
        y_prev = y_t

    return torch.stack(preds, dim=1)

In [4]:
criterion = nn.MSELoss()

pred = train_forward(model, x_input, actuator, y_true)
loss = criterion(pred, y_true)

NameError: name 'model' is not defined

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

# ----------------------------
# Model
# ----------------------------
class LSTMForecaster(nn.Module):
    def __init__(self, d_in, d_act, hidden_size, d_out):
        super().__init__()

        self.enc = nn.LSTM(d_in, hidden_size, batch_first=True)
        self.dec = nn.LSTM(d_out + d_act, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, d_out)

    def encode(self, x):
        _, (h, c) = self.enc(x)
        return h, c

    def step(self, x, state):
        out, state = self.dec(x, state)
        y = self.fc(out[:, 0])
        return y, state


# ----------------------------
# Dummy data generator
# ----------------------------
def make_dummy_data(B=32, T_in=10, T_out=15, D=4):
    """
    Simple system:
    y_t = 0.8*y_{t-1} + 0.1*actuator + noise
    """
    x = torch.randn(B, T_in, D)
    actuator = torch.randn(B, T_out, D)
    y = torch.zeros(B, T_out, D)

    y_prev = x[:, -1]

    for t in range(T_out):
        y_prev = 0.8 * y_prev + 0.1 * actuator[:, t] + 0.05 * torch.randn_like(y_prev)
        y[:, t] = y_prev

    return x, actuator, y


# ----------------------------
# Training step
# ----------------------------
def train_step(model, x, actuator, y_true, teacher_forcing=True):
    h, c = model.encode(x)

    y_prev = x[:, -1]
    preds = []

    for t in range(y_true.shape[1]):

        if teacher_forcing:
            y_in = y_true[:, t]
        else:
            y_in = y_prev

        dec_in = torch.cat([y_in, actuator[:, t]], dim=-1).unsqueeze(1)

        y_pred, (h, c) = model.step(dec_in, (h, c))

        preds.append(y_pred)
        y_prev = y_pred

    return torch.stack(preds, dim=1)


# ----------------------------
# Evaluation (no ground truth)
# ----------------------------
def evaluate(model, x, actuator):
    h, c = model.encode(x)

    y_prev = x[:, -1]
    preds = []

    for t in range(actuator.shape[1]):

        dec_in = torch.cat([y_prev, actuator[:, t]], dim=-1).unsqueeze(1)

        y_pred, (h, c) = model.step(dec_in, (h, c))

        preds.append(y_pred)
        y_prev = y_pred

    return torch.stack(preds, dim=1)


# ----------------------------
# Main
# ----------------------------
def main():

    torch.manual_seed(0)

    B, T_in, T_out, D = 64, 10, 15, 4

    x, actuator, y = make_dummy_data(B, T_in, T_out, D)

    model = LSTMForecaster(d_in=D, d_act=D, hidden_size=64, d_out=D)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # training loop
    for epoch in range(200):

        model.train()

        pred = train_step(model, x, actuator, y, teacher_forcing=True)

        loss = loss_fn(pred, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        if epoch % 20 == 0:
            model.eval()
            with torch.no_grad():
                test_pred = evaluate(model, x, actuator)
                test_loss = loss_fn(test_pred, y)

            print(f"Epoch {epoch:03d} | train {loss.item():.4f} | eval {test_loss.item():.4f}")


if __name__ == "__main__":
    main()

Epoch 000 | train 0.1430 | eval 0.1430
Epoch 020 | train 0.0970 | eval 0.1080
Epoch 040 | train 0.0446 | eval 0.0577
Epoch 060 | train 0.0136 | eval 0.0190
Epoch 080 | train 0.0047 | eval 0.0130
Epoch 100 | train 0.0030 | eval 0.0132
Epoch 120 | train 0.0025 | eval 0.0131
Epoch 140 | train 0.0023 | eval 0.0135
Epoch 160 | train 0.0021 | eval 0.0142
Epoch 180 | train 0.0019 | eval 0.0149


In [7]:
torch.manual_seed(0)

B, T_in, T_out, D = 64, 10, 15, 4

x, actuator, y = make_dummy_data(B, T_in, T_out, D)

model = LSTMForecaster(d_in=D, d_act=D, hidden_size=64, d_out=D)
opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

# training loop
for epoch in range(200):

    model.train()

    pred = train_step(model, x, actuator, y, teacher_forcing=True)

    loss = loss_fn(pred, y)

    opt.zero_grad()
    loss.backward()
    opt.step()

    if epoch % 20 == 0:
        model.eval()
        with torch.no_grad():
            test_pred = evaluate(model, x, actuator)
            test_loss = loss_fn(test_pred, y)

        print(f"Epoch {epoch:03d} | train {loss.item():.4f} | eval {test_loss.item():.4f}")

Epoch 000 | train 0.1430 | eval 0.1430
Epoch 020 | train 0.0970 | eval 0.1080
Epoch 040 | train 0.0446 | eval 0.0577
Epoch 060 | train 0.0136 | eval 0.0190
Epoch 080 | train 0.0047 | eval 0.0130
Epoch 100 | train 0.0030 | eval 0.0132
Epoch 120 | train 0.0025 | eval 0.0131
Epoch 140 | train 0.0023 | eval 0.0135
Epoch 160 | train 0.0021 | eval 0.0142
Epoch 180 | train 0.0019 | eval 0.0149


In [8]:
x.shape

torch.Size([64, 10, 4])

In [9]:
actuator.shape


torch.Size([64, 15, 4])

In [10]:
y.shape

torch.Size([64, 15, 4])